In [4]:
import os
from typing import Dict, List

from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()


class ContextBuilder:

    def __init__(self):

        client = MongoClient(os.getenv("MONGO_URI"))
        db = client[os.getenv("MONGODB_DATABASE")]

        self.section_plan_collection = db[
            os.getenv("TENDER_SECTION_PLAN_COLLECTION")
        ]

        self.requirement_collection = db[
            os.getenv("DEDUPLICATION_REQUIREMENTS_COLLECTION")
        ]

        self.evidence_collection = db[
            os.getenv("EVIDENCE_SUMMARY_COLLECTION")
        ]

        self.win_theme_collection = db[
            os.getenv("WIN_THEME_COLLECTION")
        ]

        self.evaluation_collection = db[
            os.getenv("EVALUATION_CRITERIA_COLLECTION")
        ]

    def build_context(
        self,
        company_id: str,
        tender_id: str,
        project_id: str,
        section_id: str,
    ) -> Dict:

        section = self.section_plan_collection.find_one(
            {
                "CompanyId": company_id,
                "TenderId": tender_id,
                "ProjectId": project_id,
                "Output.Sections.SectionId": section_id,
            }
        )

        if not section:
            raise Exception("Section not found")

        section_data = next(
            s
            for s in section["Output"]["Sections"]
            if s["SectionId"] == section_id
        )

        requirement_ids = section_data.get("RequirementIds", [])

        requirements = self.get_requirements(
            company_id,
            tender_id,
            project_id,
            requirement_ids,
        )

        evidence = self.get_evidence(
            company_id,
            tender_id,
            project_id,
            requirement_ids,
        )

        win_themes = self.get_win_themes(
            company_id,
            tender_id,
            project_id,
            requirement_ids,
        )

        evaluation = self.get_evaluation_criteria(
            company_id,
            tender_id,
            project_id,
            requirement_ids,
        )

        return {
            "CompanyId": company_id,
            "TenderId": tender_id,
            "ProjectId": project_id,
            "Section": section_data,
            "Requirements": requirements,
            "EvidenceSummary": evidence,
            "WinThemes": win_themes,
            "EvaluationCriteria": evaluation,
        }

    def get_requirements(
        self,
        company_id,
        tender_id,
        project_id,
        requirement_ids,
    ):

        document = self.requirement_collection.find_one(
            {
                "CompanyId": company_id,
                "TenderId": tender_id,
                "ProjectId": project_id,
            }
        )

        if not document:
            return []

        return [
            r
            for r in document["Output"]["DeduplicatedRequirements"]
            if r["DeduplicatedRequirementId"] in requirement_ids
        ]

    def get_evidence(
        self,
        company_id,
        tender_id,
        project_id,
        requirement_ids,
    ):

        cursor = self.evidence_collection.find(
            {
                "CompanyId": company_id,
                "TenderId": tender_id,
                "ProjectId": project_id,
            }
        )

        result = []

        for doc in cursor:
            req_id = doc.get("RequirementId")

            if req_id in requirement_ids:
                result.append(doc)

        return result

    def get_win_themes(
        self,
        company_id,
        tender_id,
        project_id,
        requirement_ids,
    ):

        cursor = self.win_theme_collection.find(
            {
                "CompanyId": company_id,
                "TenderId": tender_id,
                "ProjectId": project_id,
            }
        )

        result = []

        for doc in cursor:

            ids = doc.get("RequirementIds", [])

            if any(i in requirement_ids for i in ids):
                result.append(doc)

        return result

    def get_evaluation_criteria(
        self,
        company_id,
        tender_id,
        project_id,
        requirement_ids,
    ):

        cursor = self.evaluation_collection.find(
            {
                "CompanyId": company_id,
                "TenderId": tender_id,
                "ProjectId": project_id,
            }
        )

        result = []

        for doc in cursor:

            ids = doc.get("RequirementIds", [])

            if any(i in requirement_ids for i in ids):
                result.append(doc)

        return result
    
    
    
    
builder = ContextBuilder()

context = builder.build_context(
    company_id="6a45059ff419103201431533",
    tender_id="6a4506d2f4191032014315fd",
    project_id="SEMANTIC-SMOKE-001",
    section_id="sec-003"  # Replace with an actual SectionId from TenderSectionPlan
)

print(context)

Exception: Section not found

In [ ]:
!pip install pymongo

In [25]:
from pymongo import MongoClient
from typing import List, Dict, Any
import os
from dotenv import load_dotenv

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")
DATABASE_NAME = os.getenv("DATABASE_NAME")

def get_tender_sections(
    db_name: str, 
    collection_name: str, 
    tender_id: str, 
    company_id: str, 
    mongo_uri: str = "mongodb://localhost:27017/"
) -> List[Dict[str, Any]]:

    # 1. Establish connection and fetch document
    client = MongoClient(mongo_uri)
    db = client[db_name]
    collection = db[collection_name]
    
    document = collection.find_one(
        {
            "TenderId": tender_id,
            "CompanyId": company_id
        },
        {
            "_id": 0,
            "FinalJson.ProposalGroups.Sections": 1,
            "FinalJson.ProposalGroups.GroupName": 1  # Included to safely fetch GroupName
        }
    )
    
    # Close connection helper
    client.close()
    
    sections = []
    
    # 2. Extract and flatten data if document exists
    if document:
        proposal_groups = document.get("FinalJson", {}).get("ProposalGroups", [])
    
        for group in proposal_groups:
            group_name = group.get("GroupName")
            
            for section in group.get("Sections", []):
                # Inject GroupName into the section dictionary
                section["GroupName"] = group_name
                sections.append(section)
                
    return sections

# # --- How to use it ---
# if __name__ == "__main__":
#     TENDER_ID = "6a4506d2f4191032014315fd"
#     COMPANY_ID = "6a45059ff419103201431533"

#     results = get_tender_sections(
#         db_name="DocAI",
#         collection_name="TenderSectionPlan",
#         tender_id=TENDER_ID,
#         company_id=COMPANY_ID
#     )

#     print(results)



def get_sections_with_evidence_summary(
    sections: List[Dict[str, Any]],
    tender_id: str,
    company_id: str
) -> List[Dict[str, Any]]:

    client = MongoClient(MONGO_URI)

    db = client[os.getenv("MONGODB_DATABASE")]
    evidence_collection = db[os.getenv("EVIDENCE_SUMMARY_COLLECTION")]

    for section in sections:

        requirement_ids = section.get("RequirementIds", [])

        if not requirement_ids:
            section["EvidenceSummary"] = []
            continue

        pipeline = [
            {
                "$match": {
                    "TenderId": tender_id,
                    "CompanyId": company_id
                }
            },
            {
                "$unwind": "$FinalJson.CanonicalRequirements"
            },
            {
                "$match": {
                    "FinalJson.CanonicalRequirements.CanonicalRequirementId": {
                        "$in": requirement_ids
                    }
                }
            },
            {
                "$replaceRoot": {
                    "newRoot": "$FinalJson.CanonicalRequirements"
                }
            }
        ]

        section["EvidenceSummary"] = list(
            evidence_collection.aggregate(pipeline)
        )

    client.close()

    return sections



sections = get_tender_sections(
    db_name=os.getenv("MONGODB_DATABASE"),
    collection_name=os.getenv("TENDER_SECTION_PLAN_COLLECTION"),
    tender_id="6b5506d2f419103201431699",
    company_id="6b55059ff419103201431755"
)

result = get_sections_with_evidence_summary(
    sections=sections,
    tender_id="6b5506d2f419103201431699",
    company_id="6b55059ff419103201431755"
)

print(result)



[{'SectionId': 'sec-001', 'SectionName': 'Mandatory Declarations', 'SectionDescription': "Provide mandatory declarations confirming the provider's acceptance of being appointed to the Dynamic Purchasing System (DPS) for Global IT Infrastructure Services as per compliance and legal requirements.", 'Required': True, 'Priority': 'High', 'RequirementIds': ['CR5', 'CR8'], 'EvaluationCriteria': [], 'SubSections': [{'SubSectionId': 'sub-001a', 'SubSectionName': 'Legal Compliance Statements', 'SubSectionDescription': 'Detailed legal compliance declarations including GDPR, Data Protection Act, and IT security certifications.', 'RequirementIds': ['CR5a', 'CR5b'], 'EvidenceRequired': True, 'CaseStudyRequired': False}, {'SubSectionId': 'sub-001b', 'SubSectionName': 'Conflict of Interest Declarations', 'SubSectionDescription': 'Declarations regarding any potential conflicts of interest with the contracting authority.', 'RequirementIds': ['CR5c'], 'EvidenceRequired': True, 'CaseStudyRequired': False

In [ ]:

import os
from typing import Dict, List
from app.agents.Section_writer.schemas.state import ProposalGenerationState
from pymongo import MongoClient
from dotenv import load_dotenv
from typing import List, Dict, Any
load_dotenv()

import os
from dotenv import load_dotenv

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")
DATABASE_NAME = os.getenv("DATABASE_NAME")

# def get_tender_sections(
#     db_name: str, 
#     collection_name: str, 
#     tender_id: str, 
#     company_id: str, 
#     mongo_uri: str = "mongodb://localhost:27017/"
# ) -> List[Dict[str, Any]]:
#     """
#     Fetches a specific tender document and flattens its sections 
#     while injecting the parent GroupName into each section.
#     """
#     # 1. Establish connection and fetch document
#     client = MongoClient(mongo_uri)
#     db = client[db_name]
#     collection = db[collection_name]
    
#     document = collection.find_one(
#         {
#             "TenderId": tender_id,
#             "CompanyId": company_id
#         },
#         {
#             "_id": 0,
#             "FinalJson.ProposalGroups.Sections": 1,
#             "FinalJson.ProposalGroups.GroupName": 1  # Included to safely fetch GroupName
#         }
#     )
    
#     # Close connection helper
#     client.close()
    
#     sections = []
    
#     # 2. Extract and flatten data if document exists
#     if document:
#         proposal_groups = document.get("FinalJson", {}).get("ProposalGroups", [])
    
#         for group in proposal_groups:
#             group_name = group.get("GroupName")
            
#             for section in group.get("Sections", []):
#                 # Inject GroupName into the section dictionary
#                 section["GroupName"] = group_name
#                 sections.append(section)
                
#     return sections


import os
from typing import List, Dict, Any
from pymongo import MongoClient


class ContextBuilder:

    def __init__(self):
        self.client = MongoClient(os.getenv("MONGO_URI"))

        self.db = self.client[os.getenv("MONGODB_DATABASE")]

        self.section_collection = self.db[
            os.getenv("TENDER_SECTION_PLAN_COLLECTION")
        ]

        self.evidence_collection = self.db[
            os.getenv("EVIDENCE_SUMMARY_COLLECTION")
        ]

        self.docai_collection = self.db[
            os.getenv("DOCAI_COLLECTION", "DocAI")
        ]

    def close(self):
        self.client.close()

    def get_tender_sections(
        self,
        tender_id: str,
        company_id: str
    ) -> List[Dict[str, Any]]:

        document = self.section_collection.find_one(
            {
                "TenderId": tender_id,
                "CompanyId": company_id
            },
            {
                "_id": 0,
                "FinalJson.ProposalGroups.Sections": 1,
                "FinalJson.ProposalGroups.GroupName": 1
            }
        )

        sections = []

        if not document:
            return sections

        proposal_groups = (
            document.get("FinalJson", {})
            .get("ProposalGroups", [])
        )

        for group in proposal_groups:

            group_name = group.get("GroupName")

            for section in group.get("Sections", []):

                section["GroupName"] = group_name

                sections.append(section)

        return sections

    def add_evidence_summary(
        self,
        sections: List[Dict[str, Any]],
        tender_id: str,
        company_id: str
    ) -> List[Dict[str, Any]]:

        for section in sections:

            requirement_ids = section.get("RequirementIds", [])

            if not requirement_ids:
                section["EvidenceSummary"] = []
                continue

            pipeline = [
                {
                    "$match": {
                        "TenderId": tender_id,
                        "CompanyId": company_id
                    }
                },
                {
                    "$unwind": "$FinalJson.CanonicalRequirements"
                },
                {
                    "$match": {
                        "FinalJson.CanonicalRequirements.CanonicalRequirementId": {
                            "$in": requirement_ids
                        }
                    }
                },
                {
                    "$replaceRoot": {
                        "newRoot": "$FinalJson.CanonicalRequirements"
                    }
                }
            ]

            section["EvidenceSummary"] = list(
                self.evidence_collection.aggregate(pipeline)
            )

        return sections

    def get_win_themes(
        self,
        company_id: str
    ) -> List[Dict[str, Any]]:
        """
        Returns all generated win themes for a company from DocAI collection.
        """

        pipeline = [
            {
                "$match": {
                    "generated_themes.company_id": company_id
                }
            },
            {
                "$unwind": "$generated_themes"
            },
            {
                "$match": {
                    "generated_themes.company_id": company_id
                }
            },
            {
                "$replaceRoot": {
                    "newRoot": "$generated_themes"
                }
            }
        ]

        return list(self.docai_collection.aggregate(pipeline))

    def build_context(
        self,
        tender_id: str,
        company_id: str
    ) -> Dict[str, Any]:

        sections = self.get_tender_sections(
            tender_id=tender_id,
            company_id=company_id
        )

        sections = self.add_evidence_summary(
            sections=sections,
            tender_id=tender_id,
            company_id=company_id
        )

        return {
            "Sections": sections,
            "WinThemes": self.get_win_themes(company_id=company_id),
        }

###################### OUTPUT EXAMPLE ######################


#   {
#     "SectionId": "sec-001",
#     "SectionName": "Mandatory Declarations",
#     "SectionDescription": "Provide mandatory declarations confirming the provider’s acceptance of being appointed to the Dynamic Purchasing System (DPS) for Vehicle Telematics & Journey Recorders as per the compliance and legal requirements outlined in the tender.",
#     "Required": true,
#     "Priority": "High",
#     "RequirementIds": [
#       "CR5"
#     ],
#     "EvaluationCriteria": [],
#     "SubSections": [],
#     "EvidenceRequired": false,
#     "CaseStudyRequired": false,
#     "Dependencies": [],
#     "AutoCreated": false,
#     "AutoCreatedReason": "None",
#     "GroupName": "None"
#   }


############################################################



# def get_sections_with_evidence_summary(
#     sections: List[Dict[str, Any]],
#     tender_id: str,
#     company_id: str
# ) -> List[Dict[str, Any]]:

#     client = MongoClient(MONGO_URI)

#     db = client[os.getenv("MONGODB_DATABASE")]
#     evidence_collection = db[os.getenv("EVIDENCE_SUMMARY_COLLECTION")]

#     for section in sections:

#         requirement_ids = section.get("RequirementIds", [])

#         if not requirement_ids:
#             section["EvidenceSummary"] = []
#             continue

#         pipeline = [
#             {
#                 "$match": {
#                     "TenderId": tender_id,
#                     "CompanyId": company_id
#                 }
#             },
#             {
#                 "$unwind": "$FinalJson.CanonicalRequirements"
#             },
#             {
#                 "$match": {
#                     "FinalJson.CanonicalRequirements.CanonicalRequirementId": {
#                         "$in": requirement_ids
#                     }
#                 }
#             },
#             {
#                 "$replaceRoot": {
#                     "newRoot": "$FinalJson.CanonicalRequirements"
#                 }
#             }
#         ]

#         section["EvidenceSummary"] = list(
#             evidence_collection.aggregate(pipeline)
#         )

#     client.close()

#     return sections

# sections = get_tender_sections(
#     db_name=os.getenv("DATABASE_NAME"),
#     collection_name=os.getenv("TENDER_SECTION_COLLECTION"),
#     tender_id="6a1d4ca365788fd338df68dc",
#     company_id="6a1d37b5b500467c0a6f03c5"
# )

# result = get_sections_with_evidence_summary(
#     sections=sections,
#     tender_id="6a1d4ca365788fd338df68dc",
#     company_id="6a1d37b5b500467c0a6f03c5"
# )

# from pprint import pprint
# pprint(result)


############################################ Testing #####################################################

builder = ContextBuilder()

generation_context=context = builder.build_context(
    tender_id="6b5506d2f419103201431699",
    company_id="6b55059ff419103201431755",
)

print("=" * 80)
print("Context Keys")
print(context.keys())

print("=" * 80)
print("Number of Sections")
print(len(context["Sections"]))

print("=" * 80)
print("Number of Win Themes")
print(len(context["WinThemes"]))

print("=" * 80)
print("First Section")
if context["Sections"]:
    pprint(context["Sections"][0])

print("=" * 80)
print("First Win Theme")
if context["WinThemes"]:
    pprint(context["WinThemes"][0])

builder.close()

Context Keys
dict_keys(['Sections', 'WinThemes'])
Number of Sections
9
Number of Win Themes
0
First Section
{'AutoCreated': False,
 'AutoCreatedReason': None,
 'CaseStudyRequired': False,
 'Dependencies': [],
 'EvaluationCriteria': [],
 'EvidenceRequired': True,
 'EvidenceSummary': [{'CanonicalRequirement': 'Providers are to be appointed '
                                              'onto a Dynamic Purchasing '
                                              'System (DPS) for the provision '
                                              'of Vehicle Telematics & Journey '
                                              'Recorders.',
                      'CanonicalRequirementId': 'CR5',
                      'EvidenceConfidence': 0,
                      'EvidenceFound': False,
                      'EvidenceReason': 'The supplied evidence does not '
                                        'contain any statements that confirm '
                                        'appointment onto the

In [27]:
import asyncio
import json

from app.agents.Section_writer.services.mongo_services.mongo_services import (
    ContextBuilder,
)

from app.agents.Section_writer.graph.node.section_writer import (
    section_writer,
)


async def main():

    builder = ContextBuilder()

    # Step 1 : Build Context
    context = builder.build_context(
        company_id="6b55059ff419103201431755",
        tender_id="6b5506d2f419103201431699",
    )

    builder.close()

    print("=" * 100)
    print("CONTEXT BUILT")
    print("=" * 100)

    print(f"Total Sections : {len(context['Sections'])}")
    print(f"Total Win Themes : {len(context['WinThemes'])}")

    if not context["Sections"]:
        print("No sections found.")
        return

    # Step 2 : First Section
    generation_context = {
        "Section": context["Sections"][0],
        "WinThemes": context["WinThemes"],
    }

    print("\n")
    print("=" * 100)
    print("SECTION SENT TO LLM")
    print("=" * 100)

    print(
        json.dumps(
            generation_context,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    )

    # Step 3 : Call Section Writer
    result = await section_writer(
        generation_context=generation_context
    )

    print("\n")
    print("=" * 100)
    print("SECTION WRITER RESPONSE")
    print("=" * 100)

    print(
        json.dumps(
            result["response"],
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    )

    print("\n")
    print("=" * 100)
    print("TOKEN USAGE")
    print("=" * 100)

    print(result["token_usage"])


if __name__ == "__main__":
    asyncio.run(main())

ModuleNotFoundError: No module named 'langchain_groq'

In [1]:
import os
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("❌ API key is missing")
else:
    print("✅ API key is loaded")
    print("Key preview:", api_key[:8] + "..." + api_key[-4:])

client = OpenAI(api_key=api_key)

try:
    response = client.responses.create(
        model="gpt-4.1",
        input="Say hello"
    )

    print("✅ API key is valid and API call succeeded")
    print(response.output_text)

except Exception as e:
    print("❌ API call failed")
    print(e)

✅ API key is loaded
Key preview: sk-proj-...5r4A
❌ API call failed
Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


In [ ]:
import os
from datetime import datetime, timezone

from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()


class ProposalSummaryRepository:
    def __init__(self):
        mongo_uri = os.getenv("MONGO_URI")
        database_name = os.getenv("MONGODB_DATABASE") 
        collection_name = os.getenv("TENDER_PROPOSAL_SUMMARY_COLLECTION")

        if not mongo_uri:
            raise ValueError("MONGO_URI is not configured.")

        if not database_name:
            raise ValueError("MONGODB_DATABASE or MONGO_DB_NAME is not configured.")

        if not collection_name:
            raise ValueError("TENDER_PROPOSAL_SUMMARY_COLLECTION is not configured.")

        self.client = MongoClient(mongo_uri)
        self.collection = self.client[database_name][collection_name]

    def close(self):
        self.client.close()

    def save_proposal_summary(
        self,
        *,
        response: dict,
        is_regenerate: bool,
    ):
        """
        Saves the generated proposal summary.

        Status:
            Active        -> is_regenerate=False
            Regenerating  -> is_regenerate=True
        """

        status = "Regenerating" if is_regenerate else "Active"
        now = datetime.now(timezone.utc)
        proposal_plan_id = response.get("proposal_plan_id")
        proposal_id = response.get("proposal_id") or proposal_plan_id

        if not proposal_plan_id:
            raise ValueError("proposal_plan_id is required to save proposal summary.")

        document = {
            "companyId": response["company_id"],
            "tenderId": response["tender_id"],
             "versionId": response["version_id"], 
            "proposalId": proposal_id,
            "proposalPlanId": proposal_plan_id,
            "userId": response["user_id"],
            "userName": response["user_name"],
            "projectId": response["project_id"],
            "createdBy": response["user_name"],
            "CreatedBy": response["user_name"],
            "status": status,
            "isRegenerate": is_regenerate,
            "sections": [],
            "createdAt": now,
            "updatedAt": now,
        }

        for section in response.get("section_results", []):
            generated = section.get("generated_content")

            if not generated:
                continue

            document["sections"].append({
                "sectionId": generated.get("SectionId") or section.get("section_id"),
                "sectionName": generated.get("SectionName") or section.get("section_name"),
                "generatedContent": generated.get("GeneratedContent"),
                "generatedSubSections": generated.get("GeneratedSubSections", []),
                "evaluationCriteria": generated.get("EvaluationCriteria", []),
                "internalReview": generated.get("InternalReview", {}),
            })

        if is_regenerate:
            self.collection.insert_one(document)
            return document["proposalId"]

        filter_query = {
            "companyId": document["companyId"],
            "tenderId": document["tenderId"],
            "proposalPlanId": document["proposalPlanId"],
        }

        set_on_insert = {
            
            "companyId": document["companyId"],
            "tenderId": document["tenderId"],
            "proposalId": document["proposalId"],
            "proposalPlanId": document["proposalPlanId"],
            "versionId": document["versionId"],
            "createdAt": document["createdAt"],
        }

        self.collection.update_one(
            filter_query,
            {
                "$set": {
                    "userId": document["userId"],
                    "userName": document["userName"],
                    "projectId": document["projectId"],
                    "versionId": document["versionId"],
                    # "createdBy": document["createdBy"],
                    "CreatedBy": document["CreatedBy"],
                    "status": document["status"],
                    "isRegenerate": document["isRegenerate"],
                    "sections": document["sections"],
                    "updatedAt": document["updatedAt"],
                },
                "$setOnInsert": set_on_insert,
            },
            upsert=True,
        )
        return None


In [ ]:
import json

with open("agent_response.json", "r", encoding="utf-8") as f:
    api_response = json.load(f)

response = api_response["data"]